# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.0 MB/s eta 0:00:00
dependencies ok


In [4]:
import re
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference,helper,numpy_helper,TensorProto

In [5]:
TASK_ID = "task377"
TASK_TYPE = 'nonlocal nested geometry'
SYMBOLIC_RULE = 'nested rectangles compressed to Chebyshev rings'
TASK_CANDIDATES = [
    Path(COMPETITION) / f"{TASK_ID}.json",
    Path("/mnt/data") / f"{TASK_ID}.json",
    Path.cwd() / f"{TASK_ID}.json",
]
TASK_PATH = next((p for p in TASK_CANDIDATES if p.exists()), TASK_CANDIDATES[0])
OUT_DIR = Path.cwd() / f"{TASK_ID}_static_rule_onnx"
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"
SUBMISSION_PATH = Path.cwd() / "submission.zip"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FORBIDDEN_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
INPUT_SHAPE = [1, 10, 30, 30]
OUTPUT_SHAPE = [1, 10, 30, 30]

with TASK_PATH.open() as f:
    task = json.load(f)

print("task path:", TASK_PATH)
print({k: len(v) for k, v in task.items()})


task path: /kaggle/input/competitions/neurogolf-2026/task377.json
{'train': 3, 'test': 1, 'arc-gen': 262}


In [6]:

def encode_grid(grid):
    arr = np.zeros((1, 10, 30, 30), dtype=np.float32)
    h, w = len(grid), len(grid[0])
    assert h <= 30 and w <= 30, f"grid {h}x{w} cannot fit static 30x30 tensor"
    for r, row in enumerate(grid):
        for c, value in enumerate(row):
            arr[0, value, r, c] = 1.0
    return arr


def expected_tensor(grid):
    return encode_grid(grid)


def decode_tensor(tensor, h, w):
    return tensor[0, :, :h, :w].argmax(axis=0).astype(int).tolist()


def tensor_exact_checks(y, input_grid, output_grid):
    x = encode_grid(input_grid)
    input_active = (x.sum(axis=1, keepdims=True) > 0).astype(np.float32)
    outside_input_zero = bool(np.all(y * (1.0 - input_active) == 0.0))

    oh, ow = len(output_grid), len(output_grid[0])
    output_region = np.zeros((1, 1, 30, 30), dtype=np.float32)
    output_region[:, :, :oh, :ow] = 1.0
    outside_output_zero = bool(np.all(y * (1.0 - output_region) == 0.0))
    inside_sum = y[:, :, :oh, :ow].sum(axis=1)
    inside_one_hot = bool(np.all(inside_sum == 1.0))
    binary = bool(np.all((y == 0.0) | (y == 1.0)))
    return {
        "outside_input_active_canvas_zero": outside_input_zero,
        "outside_expected_output_zero": outside_output_zero,
        "inside_output_one_hot": inside_one_hot,
        "binary_tensor": binary,
    }


In [7]:
MAX = 30

class ArcBase(nn.Module):
    def __init__(self):
        super().__init__()
        rr=torch.arange(MAX,dtype=torch.float32).view(1,1,MAX,1).expand(1,1,MAX,MAX)
        cc=torch.arange(MAX,dtype=torch.float32).view(1,1,1,MAX).expand(1,1,MAX,MAX)
        self.register_buffer("rr",rr)
        self.register_buffer("cc",cc)
        self.register_buffer("rvec",torch.arange(MAX,dtype=torch.float32).view(1,1,MAX))
        self.register_buffer("cvec",torch.arange(MAX,dtype=torch.float32).view(1,1,MAX))
        self.register_buffer("color_ids",torch.arange(10,dtype=torch.float32).view(1,10))
    def active_info(self,x):
        active=(x.sum(1,keepdim=True)>0.5).to(x.dtype)
        row=(active.sum(3)>0.5).to(x.dtype) # B,1,30
        col=(active.sum(2)>0.5).to(x.dtype)
        H=row.sum(2,keepdim=True) # B,1,1
        W=col.sum(2,keepdim=True)
        return active,row,col,H,W
    def bbox(self,mask):
        # mask B,1,30,30
        rp=(mask.sum(3)>0.5).to(mask.dtype)
        cp=(mask.sum(2)>0.5).to(mask.dtype)
        before_r=torch.cumsum(rp,dim=2)-rp
        before_c=torch.cumsum(cp,dim=2)-cp
        first_r=rp*(before_r<0.5).to(mask.dtype)
        first_c=cp*(before_c<0.5).to(mask.dtype)
        after_r=torch.flip(torch.cumsum(torch.flip(rp,[2]),dim=2),[2])-rp
        after_c=torch.flip(torch.cumsum(torch.flip(cp,[2]),dim=2),[2])-cp
        last_r=rp*(after_r<0.5).to(mask.dtype)
        last_c=cp*(after_c<0.5).to(mask.dtype)
        top=(first_r*self.rvec).sum(2,keepdim=True)
        left=(first_c*self.cvec).sum(2,keepdim=True)
        bottom=(last_r*self.rvec).sum(2,keepdim=True)
        right=(last_c*self.cvec).sum(2,keepdim=True)
        present=(mask.sum((2,3),keepdim=False)>0.5).to(mask.dtype) # B,1
        return top,left,bottom,right,present
    def rect_mask(self,top,left,bottom,right):
        # scalars B,1,1; return B,1,30,30
        t=top.view(-1,1,1,1);l=left.view(-1,1,1,1);b=bottom.view(-1,1,1,1);r=right.view(-1,1,1,1)
        return ((self.rr>=t)&(self.rr<=b)&(self.cc>=l)&(self.cc<=r)).to(self.rr.dtype)
    def top_left_rect(self,h,w):
        return ((self.rr < h.view(-1,1,1,1)) & (self.cc < w.view(-1,1,1,1))).to(self.rr.dtype)
    def sample_shift(self,img,off_r,off_c):
        # img B,C,30,30; source coordinate output rr+off
        B=img.shape[0]
        sr=self.rr.expand(B,-1,-1,-1)+off_r.view(B,1,1,1)
        sc=self.cc.expand(B,-1,-1,-1)+off_c.view(B,1,1,1)
        gx=2.0*sc/(MAX-1)-1.0
        gy=2.0*sr/(MAX-1)-1.0
        grid=torch.cat([gx,gy],1).permute(0,2,3,1)
        return F.grid_sample(img,grid,mode="nearest",padding_mode="zeros",align_corners=True)

class Task377Model(ArcBase):
    def forward(self,x):
        active,*_=self.active_info(x)
        cur_rect=active
        valid=torch.ones((x.shape[0],1),dtype=x.dtype,device=x.device)
        colors=[]
        for _ in range(5):
            t,l,b,r,p=self.bbox(cur_rect)
            point=((self.rr==t.view(-1,1,1,1))&(self.cc==l.view(-1,1,1,1))).to(x.dtype)
            color=(x*point).sum((2,3))*valid
            colors.append(color)
            curpix=(x*color[:,:,None,None]).sum(1,keepdim=True)
            diff=cur_rect*(1.0-curpix)
            nt,nl,nb,nr,npres=self.bbox(diff)
            nvalid=valid*npres
            cur_rect=self.rect_mask(nt,nl,nb,nr)*nvalid[:,:,None,None]
            valid=nvalid
        # valid flags are not stored separately; infer color row sum
        colstack=torch.stack(colors,1) # B,5,10
        layer_valid=(colstack.sum(2)>0.5).to(x.dtype)
        k=layer_valid.sum(1,keepdim=True)
        D=2.0*k-1.0
        square=self.top_left_rect(D.unsqueeze(-1),D.unsqueeze(-1))
        d1=torch.minimum(self.rr,self.cc)
        d2=torch.minimum(D.view(-1,1,1,1)-1.0-self.rr,D.view(-1,1,1,1)-1.0-self.cc)
        dist=torch.minimum(d1,d2)
        y=torch.zeros_like(x)
        for i in range(5):
            ring=(dist==float(i)).to(x.dtype)*square
            y=y+colstack[:,i,:,None,None]*ring
        return y*active

model = Task377Model().eval()
print(model)

Task377Model()


In [8]:

def adv377():
    cases=[]
    specs=[
        (24,25,[4,7,2,7,9],[(3,4,20,21),(6,7,17,18),(9,10,15,16),(11,12,13,14)]),
        (20,22,[1,5,3,8],[(2,3,17,19),(5,6,14,16),(8,9,11,13)]),
    ]
    for H,W,seq,boxes in specs:
        a=np.full((H,W),seq[0],int)
        for col,(r0,c0,r1,c1) in zip(seq[1:],boxes):
            a[r0:r1+1,c0:c1+1]=col
        k=len(seq);D=2*k-1;out=np.zeros((D,D),int)
        for r in range(D):
            for c in range(D):out[r,c]=seq[min(r,c,D-1-r,D-1-c)]
        cases.append({"input":a.tolist(),"output":out.tolist()})
    return cases

adversarial_cases = adv377()

all_arc_gen = task.get("arc-gen", [])
eligible_arc_indices = [
    i for i, ex in enumerate(all_arc_gen)
    if len(ex["input"]) <= 30 and len(ex["input"][0]) <= 30
    and len(ex["output"]) <= 30 and len(ex["output"][0]) <= 30
]
required_holdout_n = int(math.ceil(0.60 * len(all_arc_gen))) if all_arc_gen else 0
assert len(eligible_arc_indices) >= required_holdout_n, (
    f"Only {len(eligible_arc_indices)} encodable ARC-gen cases, "
    f"but {required_holdout_n} are required for the 60% holdout."
)
split_rng = random.Random(20260710 + int(TASK_ID.replace("task", "")))
split_rng.shuffle(eligible_arc_indices)
holdout_indices = eligible_arc_indices[:required_holdout_n]
development_indices = eligible_arc_indices[required_holdout_n:]
arc_gen_holdout = [all_arc_gen[i] for i in holdout_indices]
oversize_arc_gen_indices = [
    i for i, ex in enumerate(all_arc_gen)
    if len(ex["input"]) > 30 or len(ex["input"][0]) > 30
    or len(ex["output"]) > 30 or len(ex["output"][0]) > 30
]

print("task type:", TASK_TYPE)
print("rule:", SYMBOLIC_RULE)
print("adversarial cases:", len(adversarial_cases))
print("ARC-gen total:", len(all_arc_gen))
print("ARC-gen 60% holdout:", len(arc_gen_holdout))
print("ARC-gen development remainder:", len(development_indices))
print("ARC-gen oversize/non-encodable indices:", oversize_arc_gen_indices)


def validate_torch(split_name, examples):
    ok = 0
    bad = []
    check_totals = Counter()
    with torch.no_grad():
        for i, ex in enumerate(examples):
            x = torch.from_numpy(encode_grid(ex["input"]))
            y = model(x).cpu().numpy()
            exp = expected_tensor(ex["output"])
            if np.array_equal(y, exp):
                ok += 1
            else:
                bad.append(i)
            for k, v in tensor_exact_checks(y, ex["input"], ex["output"]).items():
                check_totals[k] += int(v)
    result = {
        "split": split_name,
        "ok": ok,
        "total": len(examples),
        "bad_first10": bad[:10],
        **{k: int(v) for k, v in check_totals.items()},
    }
    print(result)
    return result


torch_train = validate_torch("train", task["train"])
torch_test = validate_torch("visible_test", task["test"])
torch_adversarial = validate_torch("adversarial", adversarial_cases)

assert torch_train["ok"] == torch_train["total"]
assert torch_test["ok"] == torch_test["total"]
assert torch_adversarial["ok"] == torch_adversarial["total"]


task type: nonlocal nested geometry
rule: nested rectangles compressed to Chebyshev rings
adversarial cases: 2
ARC-gen total: 262
ARC-gen 60% holdout: 158
ARC-gen development remainder: 104
ARC-gen oversize/non-encodable indices: []
{'split': 'train', 'ok': 3, 'total': 3, 'bad_first10': [], 'outside_input_active_canvas_zero': 3, 'outside_expected_output_zero': 3, 'inside_output_one_hot': 3, 'binary_tensor': 3}
{'split': 'visible_test', 'ok': 1, 'total': 1, 'bad_first10': [], 'outside_input_active_canvas_zero': 1, 'outside_expected_output_zero': 1, 'inside_output_one_hot': 1, 'binary_tensor': 1}
{'split': 'adversarial', 'ok': 2, 'total': 2, 'bad_first10': [], 'outside_input_active_canvas_zero': 2, 'outside_expected_output_zero': 2, 'inside_output_one_hot': 2, 'binary_tensor': 2}


In [9]:

dummy = torch.zeros(*INPUT_SHAPE, dtype=torch.float32)
dummy[:, 0, :10, :10] = 1.0

torch.onnx.export(
    model,
    dummy,
    ONNX_PATH,
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
    external_data=False,
)

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
inferred_model = shape_inference.infer_shapes(onnx_model)

ops = Counter(node.op_type for node in onnx_model.graph.node)
forbidden = sorted(set(ops) & FORBIDDEN_OPS)
onnx_size = ONNX_PATH.stat().st_size

print("ONNX path:", ONNX_PATH)
print("ONNX size:", onnx_size)
print("node count:", len(onnx_model.graph.node))
print("ops:", dict(ops))
print("forbidden:", forbidden)
print("function_count:", len(onnx_model.functions))

assert onnx_size < 1_400_000
assert not forbidden
assert len(onnx_model.functions) == 0


/tmp/ipykernel_17/2593287167.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX path: /kaggle/working/task377_static_rule_onnx/task377.onnx
ONNX size: 100397
node count: 691
ops: {'Identity': 1, 'Constant': 224, 'ReduceSum': 60, 'Greater': 24, 'Cast': 65, 'CumSum': 26, 'Sub': 34, 'Less': 28, 'Mul': 90, 'Reshape': 28, 'Equal': 15, 'And': 18, 'Unsqueeze': 32, 'Slice': 16, 'GreaterOrEqual': 8, 'LessOrEqual': 8, 'Concat': 1, 'Min': 3, 'Gather': 5, 'Add': 5}
forbidden: []
function_count: 0


In [10]:

session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
input_shape = session.get_inputs()[0].shape
output_shape = session.get_outputs()[0].shape
print("input_shape", input_shape)
print("output_shape", output_shape)
assert list(input_shape) == INPUT_SHAPE
assert list(output_shape) == OUTPUT_SHAPE


def validate_onnx(split_name, examples, compare_torch=True):
    ok = 0
    bad = []
    torch_equivalent = 0
    check_totals = Counter()
    elapsed = 0.0
    for i, ex in enumerate(examples):
        x = encode_grid(ex["input"])
        t0 = time.perf_counter()
        y = session.run(None, {"input": x})[0]
        elapsed += time.perf_counter() - t0
        exp = expected_tensor(ex["output"])
        if np.array_equal(y, exp):
            ok += 1
        else:
            bad.append(i)
        if compare_torch:
            with torch.no_grad():
                yt = model(torch.from_numpy(x)).cpu().numpy()
            torch_equivalent += int(np.array_equal(y, yt))
        for k, v in tensor_exact_checks(y, ex["input"], ex["output"]).items():
            check_totals[k] += int(v)
    result = {
        "split": split_name,
        "ok": ok,
        "total": len(examples),
        "bad_first10": bad[:10],
        "torch_onnx_equivalent": (torch_equivalent if compare_torch else None),
        "elapsed_seconds": elapsed,
        **{k: int(v) for k, v in check_totals.items()},
    }
    print(result)
    return result


onnx_train = validate_onnx("train", task["train"])
onnx_test = validate_onnx("visible_test", task["test"])
onnx_adversarial = validate_onnx("adversarial", adversarial_cases)
onnx_holdout = validate_onnx("arc_gen_60pct_holdout", arc_gen_holdout, compare_torch=False)

# Determinism check on an unseen-by-export concrete input.
det_example = adversarial_cases[0] if adversarial_cases else task["test"][0]
det_x = encode_grid(det_example["input"])
det_y1 = session.run(None, {"input": det_x})[0]
det_y2 = session.run(None, {"input": det_x})[0]
deterministic = bool(np.array_equal(det_y1, det_y2))

summary = {
    "task_id": TASK_ID,
    "task_type": TASK_TYPE,
    "symbolic_rule": SYMBOLIC_RULE,
    "model_family": "semantic feature-tree compiled to a static tensor graph",
    "input_shape": INPUT_SHAPE,
    "output_shape": OUTPUT_SHAPE,
    "onnx_size_bytes": onnx_size,
    "node_count": len(onnx_model.graph.node),
    "ops": dict(ops),
    "forbidden_ops": forbidden,
    "function_count": len(onnx_model.functions),
    "arc_gen_total": len(all_arc_gen),
    "arc_gen_eligible": len(eligible_arc_indices),
    "arc_gen_holdout_count": len(arc_gen_holdout),
    "arc_gen_holdout_fraction_of_total": (
        len(arc_gen_holdout) / len(all_arc_gen) if all_arc_gen else 0.0
    ),
    "arc_gen_holdout_indices_sha256": hashlib.sha256(
        json.dumps(sorted(holdout_indices)).encode("utf-8")
    ).hexdigest(),
    "arc_gen_oversize_indices": oversize_arc_gen_indices,
    "train": onnx_train,
    "test": onnx_test,
    "adversarial": onnx_adversarial,
    "arc_gen_60pct_holdout": onnx_holdout,
    "deterministic": deterministic,
}

print(json.dumps(summary, indent=2))

for section in ["train", "test", "adversarial", "arc_gen_60pct_holdout"]:
    assert summary[section]["ok"] == summary[section]["total"]
    if summary[section]["torch_onnx_equivalent"] is not None:
        assert summary[section]["torch_onnx_equivalent"] == summary[section]["total"]
    for check_name in [
        "outside_input_active_canvas_zero",
        "outside_expected_output_zero",
        "inside_output_one_hot",
        "binary_tensor",
    ]:
        assert summary[section][check_name] == summary[section]["total"]

assert deterministic

with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)
print("summary path:", SUMMARY_PATH)


input_shape [1, 10, 30, 30]
output_shape [1, 10, 30, 30]
{'split': 'train', 'ok': 3, 'total': 3, 'bad_first10': [], 'torch_onnx_equivalent': 3, 'elapsed_seconds': 0.0029394480000064505, 'outside_input_active_canvas_zero': 3, 'outside_expected_output_zero': 3, 'inside_output_one_hot': 3, 'binary_tensor': 3}
{'split': 'visible_test', 'ok': 1, 'total': 1, 'bad_first10': [], 'torch_onnx_equivalent': 1, 'elapsed_seconds': 0.0006909539999924164, 'outside_input_active_canvas_zero': 1, 'outside_expected_output_zero': 1, 'inside_output_one_hot': 1, 'binary_tensor': 1}
{'split': 'adversarial', 'ok': 2, 'total': 2, 'bad_first10': [], 'torch_onnx_equivalent': 2, 'elapsed_seconds': 0.0014165770000431621, 'outside_input_active_canvas_zero': 2, 'outside_expected_output_zero': 2, 'inside_output_one_hot': 2, 'binary_tensor': 2}
{'split': 'arc_gen_60pct_holdout', 'ok': 158, 'total': 158, 'bad_first10': [], 'torch_onnx_equivalent': None, 'elapsed_seconds': 0.09221848400034105, 'outside_input_active_canva

In [11]:

if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()

with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")

print("submission:", SUBMISSION_PATH)
print("zip contents:")
with zipfile.ZipFile(SUBMISSION_PATH) as zf:
    infos = zf.infolist()
    for info in infos:
        print(info.filename, info.file_size)
    assert len(infos) == 1
    assert infos[0].filename == f"{TASK_ID}.onnx"


submission: /kaggle/working/submission.zip
zip contents:
task377.onnx 100397
